In [1]:
# Imports and setup
import json
import warnings
import torch
from pprint import pprint


import pandas as pd
from sklearn.preprocessing import LabelEncoder

from benchmark.config import DATA_PROCESSED, N_TRIALS, NUM_FOLDS
from benchmark.models.xgboost_model import (
    build_pipeline_clf,
    build_preprocessor,
    evaluate_clf,
    export_results_clf,
    final_fit_clf,
    get_search_space,
    run_search_clf,
)

warnings.filterwarnings("ignore")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Main config
data_name = "sleep_health"
processed = DATA_PROCESSED / data_name
k_folds = NUM_FOLDS
n_trials = N_TRIALS

xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
# xgb_device = "cpu"
print("XGBoost device:", xgb_device)

XGBoost device: cuda


In [3]:
# Load data and metadata
train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as file:
    metadata = json.load(file)

print("Dataset path:", processed)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nMetadata:")
pprint(metadata)

Dataset path: /home/shado/college_project/comp9417/9417GroupProject/data/processed/sleep_health
Train shape: (16000, 33)
Test shape: (4000, 33)

Metadata:
{'bin_cols': ['sleep_aid_used', 'shift_work'],
 'cat_cols': ['gender',
              'occupation',
              'country',
              'chronotype',
              'mental_health_condition',
              'season',
              'day_type'],
 'num_cols': ['age',
              'bmi',
              'sleep_duration_hrs',
              'sleep_quality_score',
              'rem_percentage',
              'deep_sleep_percentage',
              'sleep_latency_mins',
              'wake_episodes_per_night',
              'caffeine_mg_before_bed',
              'alcohol_units_before_bed',
              'screen_time_before_bed_mins',
              'exercise_day',
              'steps_that_day',
              'nap_duration_mins',
              'stress_score',
              'work_hours_that_day',
              'heart_rate_resting_bpm',
       

In [4]:
# Split x and y
target_column = metadata["target_col"]

x_train_full = train_df.drop(columns=[target_column]).copy()
x_test = test_df.drop(columns=[target_column]).copy()

label_encoder = LabelEncoder()
y_train_full = label_encoder.fit_transform(train_df[target_column])
y_test = label_encoder.transform(test_df[target_column])

class_names = label_encoder.classes_
n_classes = len(class_names)

print("\nEncoded classes:")
print(dict(enumerate(class_names)))


Encoded classes:
{0: 'Healthy', 1: 'Mild', 2: 'Moderate', 3: 'Severe'}


In [5]:
# Build preprocessor and run hyperparameter search
preprocessor = build_preprocessor(metadata)
pipeline = build_pipeline_clf(preprocessor, n_classes, device=xgb_device, n_jobs=1)
search_space = get_search_space()

best_search_params, best_model_params, best_cv_f1, search_time = run_search_clf(
    pipeline,
    search_space,
    x_train_full,
    y_train_full,
    n_classes=n_classes,
    k_folds=k_folds,
    n_trials=n_trials,
)

print("\nBest CV F1:", best_cv_f1)
print("\nBest hyperparameters:")
pprint(best_model_params)


Best CV F1: 0.8501692555445217

Best hyperparameters:
{'colsample_bytree': 0.8504326788691458,
 'gamma': 0.21905332579218617,
 'learning_rate': 0.13565773665802744,
 'max_depth': 3,
 'min_child_weight': 6,
 'n_estimators': 725,
 'reg_alpha': 0.002441592339653542,
 'reg_lambda': 3.5048149824176797,
 'subsample': 0.7922903329492129}


In [6]:
# Final fit with best params
final_pipeline, fit_time = final_fit_clf(
    preprocessor,
    best_search_params,
    x_train_full,
    y_train_full,
    n_classes=n_classes,
    device=xgb_device,
)

In [7]:
# Evaluate on train and test
train_metrics = evaluate_clf(final_pipeline, x_train_full, y_train_full, n_classes)
test_metrics = evaluate_clf(final_pipeline, x_test, y_test, n_classes)

In [8]:
# Final results
print("\nFinal results:")

print("Search time (seconds):", search_time)
print("Final training time (seconds):", fit_time)
print("Best CV F1:", best_cv_f1)

print("\nTrain Accuracy:", train_metrics["accuracy"])
print("Train AUC-ROC:", train_metrics["auc_roc"])
print("Train F1:", train_metrics["f1"])
print("Train Precision:", train_metrics["precision"])
print("Train Recall:", train_metrics["recall"])
print(
    "Train inference time per sample (seconds):",
    train_metrics["infer_time_per_sample_s"],
)

print("\nTest Accuracy:", test_metrics["accuracy"])
print("Test AUC-ROC:", test_metrics["auc_roc"])
print("Test F1:", test_metrics["f1"])
print("Test Precision:", test_metrics["precision"])
print("Test Recall:", test_metrics["recall"])
print(
    "Test inference time per sample (seconds):", test_metrics["infer_time_per_sample_s"]
)


Final results:
Search time (seconds): 128.6545831780022
Final training time (seconds): 1.400232842002879
Best CV F1: 0.8501692555445217

Train Accuracy: 0.997625
Train AUC-ROC: 0.9999716813503663
Train F1: 0.9954585570952407
Train Precision: 0.9967929542465438
Train Recall: 0.994158267000593
Train inference time per sample (seconds): 3.3072505623295e-06

Test Accuracy: 0.9465
Test AUC-ROC: 0.9908594159860822
Test F1: 0.8457076235562169
Test Precision: 0.8612285098430403
Test Recall: 0.8324706988814975
Test inference time per sample (seconds): 4.7481540004810085e-06


In [9]:
# Export model and results
export_results_clf(
    data_name,
    final_pipeline,
    best_model_params,
    train_metrics,
    test_metrics,
    fit_time,
    search_time,
    best_cv_f1,
    y_test,
    class_names,
)


Model saved to /home/shado/college_project/comp9417/9417GroupProject/models/sleep_health/xgboost.joblib
Pipeline saved to /home/shado/college_project/comp9417/9417GroupProject/models/sleep_health/xgboost_pipeline.joblib
Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/xgboost.json
